In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

from conus_biomass import dir_info
from conus_biomass.make_figures import maps
from conus_biomass.process_outputs.generate_state_csv import calculate_total_carbon_stock

In [ ]:
def get_filtered_output(
    year,
    fdir=dir_info.dir_processed
    + "model_results/1000m_2026feb02_processed/",  # dir_info.dir_model_output,
    ftype="netcdf",
    vartype="predicted_biomass_filtered_",
):
    fname = fdir + vartype + str(year)
    if ftype == "zarr":
        ds = xr.open_zarr(fname + ".zarr")
    elif ftype == "netcdf":
        ds = xr.load_dataset(fname + ".nc")
    return ds


def clip_to_shape(da, shp):
    if shp.crs != da.rio.crs:
        shp = shp.to_crs(da.rio.crs)
        print("reprojected shape")
    clipped = da.rio.clip(shp.geometry, shp.crs, drop=True)
    return clipped

In [ ]:
usfs = pd.read_csv("USFS_western_stocks.csv")

In [ ]:
usfs_deltas = usfs["0"].values[1:] - usfs["0"].values[:-1]

In [ ]:
usfs_years = usfs["Unnamed: 0"].values[:-1]

In [ ]:
slope = xr.open_zarr(dir_info.dir_processed + "data_on_ref_grid/1000m/aspect.zarr")
crs = slope["spatial_ref"].crs_wkt

In [ ]:
fire_losses_west = []
years = np.arange(2005, 2023)

for year in years:
    print(year)
    biomass_forest = get_filtered_output(
        year=year, vartype="predicted_biomass_delta_burned_filtered_"
    )
    biomass_forest = biomass_forest.rio.write_crs(crs)
    biomass_forest = biomass_forest

    biomass_forest_west = clip_to_shape(
        da=biomass_forest["predicted_biomass_delta_burned"], shp=maps.SHP_WESTERN
    )
    MMT_west = calculate_total_carbon_stock(biomass_forest_west, grid_res=1000)
    fire_losses_west.append(MMT_west)

    del biomass_forest, biomass_forest_west

In [ ]:
unburned_gains_west = []

for year in years:
    print(year)
    biomass_forest = get_filtered_output(
        year=year, vartype="predicted_biomass_delta_unburned_filtered_"
    )
    biomass_forest = biomass_forest.rio.write_crs(crs)
    biomass_forest = biomass_forest

    biomass_forest_west = clip_to_shape(
        da=biomass_forest["predicted_biomass_delta_unburned"], shp=maps.SHP_WESTERN
    )
    MMT_west = calculate_total_carbon_stock(biomass_forest_west, grid_res=1000)
    unburned_gains_west.append(MMT_west)

    del biomass_forest, biomass_forest_west

In [ ]:
net_changes = np.array(unburned_gains_west) + np.array(fire_losses_west)

In [ ]:
plt.rcParams.update({"font.size": 14})
fig, axes = plt.subplots(nrows=1, ncols=1, figsize=(16 / 2, 5))

ax = axes

ax.set_ylabel("Live AGB Change per year (MMT C/year)")
ax.spines[["right", "top"]].set_visible(False)
ax.legend(frameon=False)
ax.set_xlim([2004, 2024])
ax.axhline(y=0, linestyle=":", color="gray", linewidth=1)

# ax.axvspan(2005, 2015, color="lightgray", alpha=0.3)
# ax.axvspan(2015, 2022, color="lightgray", alpha=0.3)

ax.plot(
    years, fire_losses_west, linewidth=5, color="firebrick", label="Burn losses"
)  # , marker='o', markersize=10)
ax.plot(
    years, unburned_gains_west, linewidth=5, color="lightseagreen", label="Unburned gains"
)  # , marker='o', markersize=10)
ax.plot(
    years, net_changes, linewidth=5, color="C0", label="Net change"
)  # , marker='o', markersize=10)
ax.plot(
    usfs_years, usfs_deltas, linewidth=5, color="darkgray", label="USFS/EPA net change", zorder=0
)
plt.legend(frameon=False)

rolling_avg = pd.Series(net_changes).rolling(window=5, center=True).mean()
ax.plot(
    years,
    rolling_avg,
    linewidth=3,
    color="C0",
    label="Net change (5-yr avg)",
    linestyle="--",
    alpha=0.8,
)
rolling_avg = pd.Series(np.array(unburned_gains_west)).rolling(window=5, center=True).mean()
ax.plot(
    years,
    rolling_avg,
    linewidth=3,
    color="lightseagreen",
    label="Net change (5-yr avg)",
    linestyle="--",
    alpha=0.8,
)
rolling_avg = pd.Series(np.array(fire_losses_west)).rolling(window=5, center=True).mean()
ax.plot(
    years,
    rolling_avg,
    linewidth=3,
    color="firebrick",
    label="Net change (5-yr avg)",
    linestyle="--",
    alpha=0.8,
)

plt.tight_layout()
plt.savefig("Annual_changes.png", dpi=200)
plt.xlim([2005, 2023])
# ax.plot(years,np.array(fire_losses_west)+
#        np.array(unburned_gains_west), linewidth=5, color='C1', marker='o', markersize=10)